# 21. 전체 저장 결과에서 PDF 근거 검색

[프로젝트 적용] 앞 단계에서 저장한 전체 벡터를 사용합니다. 여기서는 질문 하나만 벡터로 바꾸고 DB를 읽습니다. 기본 검색은 기존 글자 TF-IDF·질문 목적 규칙을 함께 사용합니다.
[프로젝트 추가] 처리 작업 번호(run_id)를 지정해 표본·다른 작업·다른 모델의 벡터와 섞이지 않게 합니다.

현재 단계는 **근거 검색**입니다. 검색 후보를 챗봇의 최종 답변으로 보지는 않습니다. 답변 생성·근거 부족 판단은 M5에서 연결합니다.


In [ ]:
from pathlib import Path
import sys

# 파일 실행 위치가 notebooks여도 프로젝트의 src 폴더를 찾습니다.
project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
if not (project / "src/car_search_rag/zzong_santafe_lag").is_dir():
    raise FileNotFoundError("mle-02-p1-team1 또는 그 notebooks 폴더에서 실행하세요.")
if str(project / "src") not in sys.path:
    sys.path.insert(0, str(project / "src"))


In [ ]:
import json
from car_search_rag.zzong_santafe_lag.db_search_service import DbFullSearchService

# [프로젝트 추가] 이번 실제 저장 결과를 읽습니다. 벡터나 접속 비밀번호는 이 파일에 없습니다.
report_path = project / "data/zzong_santafe_lag/reports/full_store_20261002.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
if report["phase"] != "completed" or not report.get("sample_unchanged"):
    raise ValueError("전체 저장 완료와 기존 표본 유지 결과를 먼저 확인하세요.")
run_id = report["save_result"]["run_id"]
print("전체 작업 번호:", run_id)
print("임베딩 개수·차원:", report["embedding_report"]["shape"])
print("개인 DB 할당 용량(MiB):", round(report["save_result"]["allocated_bytes"] / 1024**2, 2))
db_search = DbFullSearchService(run_id)


In [ ]:
# 질문을 바꿔 다시 실행하면 같은 로컬 모델을 재사용합니다. 문서 벡터는 재계산하지 않습니다.
question = "차대번호는 어디에서 확인하나요?"
result = db_search.search(question, top_k=3, progress=print)
for candidate in result["candidates"]:
    print(f"\n후보 {candidate['rank']}: {candidate['title']}")
    print("결합 점수:", round(candidate["ranking_score"], 3), "| 질문 목적:", candidate["routing"]["intents"])
    print("찾은 청크:", candidate["matched_chunk"])
    for context in candidate["context_records"]:
        print("PDF 쪽:", context["source_pages"], "| 매뉴얼 시작 쪽:", context["manual_page_number"])
        print("검토 상태:", context["verification_status"])
    print("연결 그림 수:", len(candidate["image_parts"]))
    print("원본 대조 필요:", candidate["pending_review_record_ids"])
print("제목만 있어 후보에서 제외한 기록 수:", result["excluded_heading_records"])
print("\n" + result["coverage_notice"])


In [ ]:
from IPython.display import Image, display

# [프로젝트 추가] 이미 업로드한 그림만 표시합니다. 아직 참조뿐인 그림을 빈 주소로 요청하지 않습니다.
# 표시한 그림이 정확히 해당 문장을 설명하는지도 검토 상태와 원본 페이지로 확인해야 합니다.
for image in result["candidates"][0]["image_parts"]:
    print("PDF 쪽:", image["pdf_page_number"], "| 파일:", image["file_name"] or image["pdf_image_key"])
    for description in image["descriptions"]:
        if description["description"]:
            print("설명:", description["description"])
    if image["public_url"]:
        display(Image(url=image["public_url"]))
    else:
        print("이 그림은 DB 참조만 연결되어 있으며 공개 파일 업로드 전입니다.")


## 다음 단계: M5

검색된 글에서 질문에 답할 수 있는 근거를 골라 페이지·그림과 함께 답변을 구성합니다. 관련 후보가 있다는 이유만으로 답할 수 있다고 판단하지 않습니다.

전체 그림 검토와 업로드는 아직입니다. 자동 초안 507개·설명 없는 연결 1,145건은 검토 대상으로 남아 있습니다. 이번 노트북은 DB 자료를 수정하지 않습니다.
